In [272]:
import copy
import os
import argparse
from erasure.utils.logger import GLogger
import torch
import numpy as np
import random
from erasure.utils.config.local_ctx import Local
from erasure.utils.config.global_ctx import Global, bcolors 
from erasure.core.factory_base import ConfigurableFactory
from erasure.data.datasets.DatasetManager import DatasetManager

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [273]:
config_file = os.path.join("configs","LinkAttack", "Citeseer_edge.jsonc")

In [274]:
global_ctx = Global(config_file)
global_ctx.factory = ConfigurableFactory(global_ctx)

#Create Dataset
data_manager = global_ctx.factory.get_object( Local( global_ctx.config.data ))
global_ctx.dataset = data_manager

@,41601115 | INFO | 2742752 - Creating Global Context for: configs/LinkAttack/Citeseer_edge.jsonc
��j�,41601117 | INFO | 2742752 - Setting seeds to: 0
!,41601118 | INFO | 2742752 - REMOVAL TYPE SET AS edge
,41601125 | INFO | 2742752 - Caching System: False.
x,41601126 | INFO | 2742752 - Instantiating: torch_geometric.datasets.Planetoid


�N�v,41601170 | INFO | 2742752 - Created Configurable: erasure.data.data_sources.TorchGeometricDataSource.TorchGeometricDataSource
�rs,41601170 | INFO | 2742752 - {'class': 'erasure.data.data_sources.TorchGeometricDataSource.TorchGeometricDataSource', 'parameters': {'datasource': {'class': 'torch_geometric.datasets.Planetoid', 'parameters': {'root': 'resources/data', 'name': 'Citeseer'}, 'preprocess': []}}}
x,41601219 | INFO | 2742752 - Instantiating: erasure.data.datasets.DataSplitterGraph.DataSplitterPercentage
0�,41601266 | INFO | 2742752 - ['all', 'all_shuffled', '-']
x,41601267 | INFO | 2742752 - Instantiating: erasure.data.datasets.DataSplitterGraph.DataSplitterPercentage
���P,41601268 | INFO | 2742752 - ['all', 'all_shuffled', '-', 'train_0', 'test']
x,41601269 | INFO | 2742752 - Instantiating: erasure.data.datasets.DataSplitterGraph.DataSplitterPercentage
���X,41601270 | INFO | 2742752 - ['all', 'all_shuffled', '-', 'train_0', 'test', 'validation', 'train']

In [275]:
#Create Predictor
current = Local(global_ctx.config.predictor)
current.dataset = data_manager
predictor = global_ctx.factory.get_object(current)
global_ctx.predictor = predictor
global_ctx.logger.info('Global Predictor: ' + str(predictor))

x,41601360 | INFO | 2742752 - Instantiating: erasure.model.graphs.GCN.GCN
x,41601368 | INFO | 2742752 - Instantiating: torch.optim.Adam
x,41601369 | INFO | 2742752 - Instantiating: torch.nn.CrossEntropyLoss


graph has edges  torch.Size([2, 9104])
��t,41601429 | INFO | 2742752 - epoch = 0 ---> loss = 1.7978	 accuracy = 0.1720
graph has edges  torch.Size([2, 9104])
��t,41601479 | INFO | 2742752 - epoch = 1 ---> loss = 1.7470	 accuracy = 0.3599
graph has edges  torch.Size([2, 9104])
��t,41601527 | INFO | 2742752 - epoch = 2 ---> loss = 1.6973	 accuracy = 0.5436
graph has edges  torch.Size([2, 9104])
��t,41601578 | INFO | 2742752 - epoch = 3 ---> loss = 1.6504	 accuracy = 0.6200
graph has edges  torch.Size([2, 9104])
��t,41601627 | INFO | 2742752 - epoch = 4 ---> loss = 1.5992	 accuracy = 0.6785
graph has edges  torch.Size([2, 9104])
��t,41601675 | INFO | 2742752 - epoch = 5 ---> loss = 1.5530	 accuracy = 0.7027
graph has edges  torch.Size([2, 9104])
��t,41601726 | INFO | 2742752 - epoch = 6 ---> loss = 1.5004	 accuracy = 0.7219
graph has edges  torch.Size([2, 9104])
��t,41601776 | INFO | 2742752 - epoch = 7 ---> loss = 1.4527	 accuracy = 0.7378
graph has edges  torch.Size([2, 9104])
��t,41601

In [276]:
data_manager.partitions['all'][0][0]

Data(x=[3327, 3703], edge_index=[2, 9104])

In [277]:
data_manager.partitions['all'][0][1]

tensor([3, 1, 5,  ..., 3, 1, 5])

In [278]:
import torch
print(torch.version.cuda)

12.6


In [279]:
print(predictor.optimizer)

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    initial_lr: 0.001
    lr: 0.0005000000000000002
    maximize: False
    weight_decay: 0
)


In [280]:
import torch
print(torch.__version__)
print(torch.version.cuda)


2.7.0+cu126
12.6


In [281]:
#Create unlearners 
unlearners = []
unlearners_cfg = global_ctx.config.unlearners
for un in unlearners_cfg:
    current = Local(un)
    current.dataset = data_manager
    current.predictor = copy.deepcopy(predictor)
    unlearners.append( global_ctx.factory.get_object(current) )

x,41604515 | INFO | 2742752 - Instantiating: torch.optim.Adam
�N�v,41604517 | INFO | 2742752 - Created Configurable: erasure.unlearners.graph_unlearners.UNSIR.UNSIR


/NFSHOME/adangelo/LinkInference/erasure/unlearners/graph_unlearners/GraphUnlearner.py:24: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.labels = torch.tensor(self.labels)


In [282]:
data_manager.partitions['all'][0][0]

Data(x=[3327, 3703], edge_index=[2, 9104])

In [283]:
data_manager.partitions['all'][0][0].edge_index

tensor([[   0,    1,    1,  ..., 3324, 3325, 3326],
        [ 628,  158,  486,  ..., 2820, 1643,   33]])

In [284]:
data_manager.partitions['all'].revise_graph_edges([(0,1),(0,2)])[0][0]

Data(x=[3327, 3703], edge_index=[2, 0])

In [285]:
print(len( data_manager.partitions['forget']) )

1820


In [286]:
#Evaluator
current = Local(global_ctx.config.evaluator)
current.unlearners = unlearners
evaluator = global_ctx.factory.get_object(current)

# Evaluations
for unlearner in unlearners:
    global_ctx.logger.info(f'''{bcolors.OKGREEN}####\t\t Evaluating Unlearner {unlearner.__class__.__name__} \t\t####{bcolors.ENDC}''')
    evaluator.evaluate(unlearner,predictor)

�N�v,41604913 | INFO | 2742752 - Created Configurable: erasure.evaluations.running.RunTime
 )NP,41604915 | INFO | 2742752 - Function: <function accuracy_score at 0x7f115905f0d0>
�N�v,41604917 | INFO | 2742752 - Created Configurable: erasure.evaluations.measures.TorchSKLearnGraph
0+NP,41604918 | INFO | 2742752 - Function: <function accuracy_score at 0x7f115905f0d0>
�N�v,41604918 | INFO | 2742752 - Created Configurable: erasure.evaluations.measures.TorchSKLearnGraph
�-NP,41604919 | INFO | 2742752 - Function: <function accuracy_score at 0x7f115905f0d0>
�N�v,41604920 | INFO | 2742752 - Created Configurable: erasure.evaluations.measures.TorchSKLearnGraph
��MP,41604921 | INFO | 2742752 - Function: <function accuracy_score at 0x7f115905f0d0>
�N�v,41604921 | INFO | 2742752 - Created Configurable: erasure.evaluations.measures.TorchSKLearnGraph
�N�v,41604922 | INFO | 2742752 - Created Configurable: erasure.evaluations.measures.SaveValues
�N�v,41604922 | INFO | 2742752 - Cre

�esv,41604961 | INFO | 2742752 - Unlearning copyed predictor: <erasure.model.TorchGraphModel.TorchGraphModel object at 0x7f11505ef820>
,41604962 | INFO | 2742752 - Starting UNSIR with 1 epochs for the impair phase
x[forget_set] shape: torch.Size([0, 3703])
noise() shape: torch.Size([0, 3703])
,41605884 | INFO | 2742752 - UNSIR - Epoch 1/1 - Avg Loss: 1.1414
�esv,41605923 | INFO | 2742752 - sklearn.metrics.accuracy_score on partition: "test", target model: unlearned, unlearned graph: True: 0.7672672672672672 of <erasure.model.TorchGraphModel.TorchGraphModel object at 0x7f11505ef820>
�esv,41605956 | INFO | 2742752 - sklearn.metrics.accuracy_score on partition: "test", target model: original, unlearned graph: True: 0.7522522522522522 of <erasure.model.TorchGraphModel.TorchGraphModel object at 0x7f117373cdf0>
�esv,41605983 | INFO | 2742752 - sklearn.metrics.accuracy_score on partition: "test", target model: unlearned, unlearned graph: False: 0.7612612612612613 of <erasure.mo